# Candidate Analysis

Two small-scale manual analyses on DeepSeek runs:

1. **`nr_of_candidates`** — how many hypothetical student solution paths the model considers in reasoning mode vs. CoT mode (15 problems each → 30 traces per dataset).
2. **`discards_valid`** — how many distractor candidates that *would* have matched one of the ground-truth final distractors are discarded by the model before it commits to a final set (30 reasoning traces per dataset, sampled from the low-match bucket where this is most informative).

This notebook (a) samples problems, (b) writes the per-problem trace files into `manual_inspection/{nr_of_candidates,discards_valid}/{dataset}/`, ready for manual annotation, and (c) parses the annotated files to compute aggregate averages. Existing files are not overwritten so prior manual annotations are preserved.

In [ ]:
import os
import re
import json
import random
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from src.datasets import get_or_create_dataset

load_dotenv()

In [ ]:
eedi_dataset = get_or_create_dataset("eedi_data", n_limit=500)
sciq_dataset = get_or_create_dataset("sciq_data", n_limit=500)

datasets_by_datafolder = {
    "eedi_data": eedi_dataset,
    "sciq_data": sciq_dataset,
}

## Sampling + dump helpers

We stratify the same way `040-annotation.ipynb` does (`proportional_match`):
- `low_match_solvable`: `solvable and proportional_match < 0.5`
- `high_match_solvable`: `solvable and proportional_match > 0.5`

(The `unsolvable` bucket from the original eedi sweep is dropped — sciq has effectively no unsolvable problems, and including it asymmetrically would muddle the comparison. 16 = 2 × 8 samples per file.)

In [ ]:
SEED = 42
SEP = "-" * 60

def _load_run(data_folder: str, run_name: str):
    """Returns (responses_dict, results_df)."""
    with open(f"{data_folder}/joint_results/{run_name}_responses_by_datapointid.json") as f:
        responses = json.load(f)
    results_df = pd.read_csv(f"{data_folder}/joint_results/{run_name}_results.csv")
    return responses, results_df

def _build_df(data_folder: str, run_name: str) -> pd.DataFrame:
    responses, results_df = _load_run(data_folder, run_name)
    dataset = datasets_by_datafolder[data_folder]
    rows = []
    for k in responses.keys():
        dp = dataset[int(k)]
        try:
            res = results_df[results_df["Id"] == int(k)].iloc[0]
        except IndexError:
            continue
        rows.append({
            "Id": int(k),
            "Question": dp["Problem"]["Question"],
            "Answer":   dp["Problem"]["Answer"],
            "Solvable": dp["Problem"].get("Solvable", True),
            "proportional_match": res["proportional_match"],
            "distractors": dp["Choices"]["Distractors"],
        })
    return pd.DataFrame(rows)

def _stratified_sample_ids(df: pd.DataFrame, buckets: dict, seed: int = SEED) -> dict:
    """buckets: {bucket_name: (mask, n)}. Returns {bucket_name: [ids]}."""
    out = {}
    rng = random.Random(seed)
    for name, (mask, n) in buckets.items():
        pool = df[mask]["Id"].tolist()
        rng.shuffle(pool)
        out[name] = pool[:n]
    return out

In [ ]:
def _format_record_nr_candidates(pid: int, question: str, reasoning: str, reasoning_label: str,
                                 distractors: str, correct_answer: str) -> str:
    return (
        f"Problem Id: {pid}\n"
        f"Question:\n{question}\n"
        f"{reasoning_label}:\n{reasoning}\n"
        f"Distractors:\n{distractors}\n"
        f"Correct Answer:\n{correct_answer}\n\n"
        f"CONSIDERED SCENARIOS: \n\n"
        f"#SCENARIOS: \n"
        f"{SEP}\n"
    )

def _format_record_discards(pid: int, question: str, reasoning: str, response: str,
                            distractors: str, correct_answer: str) -> str:
    return (
        f"Problem Id: {pid}\n"
        f"Question:\n{question}\n"
        f"Annotated Reasoning:\n{reasoning}\n"
        f"Response:\n{response}\n"
        f"Distractors:\n{distractors}\n"
        f"Correct Answer:\n{correct_answer}\n\n"
        f"DROPPED: \n"
        f"{SEP}\n"
    )

def dump_nr_candidates(data_folder: str, run_name: str, is_reasoning: bool,
                       out_path: str, n_per_bucket: int = 8, seed: int = SEED,
                       overwrite: bool = False):
    """Write a single file with stratified samples (low_match_solvable / high_match_solvable).
    For nr_of_candidates we use the model's reasoning trace (or step_by_step for CoT runs)."""
    if os.path.exists(out_path) and not overwrite:
        print(f"[skip] {out_path} already exists; pass overwrite=True to regenerate.")
        return
    responses, _ = _load_run(data_folder, run_name)
    df = _build_df(data_folder, run_name)
    buckets = {
        "low_match_solvable":   ((df["Solvable"]) & (df["proportional_match"] < 0.5),         n_per_bucket),
        "high_match_solvable":  ((df["Solvable"]) & (df["proportional_match"] > 0.5),         n_per_bucket),
    }
    sampled = _stratified_sample_ids(df, buckets, seed=seed)

    reasoning_label = "Annotated Reasoning" if is_reasoning else "Step-By-Step Reasoning and Answer"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        for bucket_name, ids in sampled.items():
            f.write(f"===== {bucket_name}_annot.csv =====\n\n")
            for pid in ids:
                row = df[df["Id"] == pid].iloc[0]
                r = responses[str(pid)]
                trace = r.get("raw_reasoning") if is_reasoning else (r.get("step_by_step") or r.get("raw", ""))
                f.write(_format_record_nr_candidates(
                    pid, row["Question"], trace or "", reasoning_label,
                    row["distractors"], row["Answer"],
                ))
    print(f"[wrote] {out_path} ({sum(len(v) for v in sampled.values())} records across {len(sampled)} buckets)")

def dump_discards_valid(data_folder: str, run_name: str, out_path: str,
                        n_samples: int = 30, seed: int = SEED, overwrite: bool = False):
    """Write a file of `n_samples` traces drawn from the low-match-solvable bucket, where the model
    most likely considered (and then discarded) candidates that match the held-out distractors."""
    if os.path.exists(out_path) and not overwrite:
        print(f"[skip] {out_path} already exists; pass overwrite=True to regenerate.")
        return
    responses, _ = _load_run(data_folder, run_name)
    df = _build_df(data_folder, run_name)
    buckets = {
        "low_match_solvable": ((df["Solvable"]) & (df["proportional_match"] < 0.5), n_samples),
    }
    sampled = _stratified_sample_ids(df, buckets, seed=seed)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        for bucket_name, ids in sampled.items():
            f.write(f"===== {bucket_name}_annot.csv =====\n\n")
            for pid in ids:
                row = df[df["Id"] == pid].iloc[0]
                r = responses[str(pid)]
                reasoning = r.get("raw_reasoning") or ""
                response = r.get("raw") or ""
                f.write(_format_record_discards(
                    pid, row["Question"], reasoning, response,
                    row["distractors"], row["Answer"],
                ))
    print(f"[wrote] {out_path} ({sum(len(v) for v in sampled.values())} records)")

## Generate samples

Runs reused:
- reasoning  → `deepseek-naive-deepseek-reasoner`
- CoT        → `deepseek-naive-cot-deepseek-chat`

Existing manually annotated files are left untouched (see `overwrite=False`).

In [ ]:
REASONER_RUN = "deepseek-naive-deepseek-reasoner"
COT_RUN      = "deepseek-naive-cot-deepseek-chat"

for data_folder in ["eedi_data", "sciq_data"]:
    dump_nr_candidates(data_folder, REASONER_RUN, is_reasoning=True,
                       out_path=f"manual_inspection/nr_of_candidates/{data_folder}/deepseek_reasoner_naive_reasoner_joint.txt")
    dump_nr_candidates(data_folder, COT_RUN,      is_reasoning=False,
                       out_path=f"manual_inspection/nr_of_candidates/{data_folder}/deepseek_chat_naive_cot_joint.txt")
    dump_discards_valid(data_folder, REASONER_RUN,
                        out_path=f"manual_inspection/discards_valid/{data_folder}/deepseek_reasoner_naive_reasoner_joint.txt")

## Parse annotated files → aggregate stats

After the dump files have been manually annotated (filling in the `#SCENARIOS:` and `DROPPED:` numeric fields), this section reads them back and reports the averages used in the paper.

In [ ]:
_SCENARIOS_RE = re.compile(r"^#SCENARIOS:\s*(\d+)", re.MULTILINE)
_DROPPED_RE   = re.compile(r"^DROPPED:\s*(\d+)",    re.MULTILINE)

def _parse_counts(path: str, pattern: re.Pattern) -> list:
    if not os.path.exists(path):
        return []
    with open(path) as f:
        text = f.read()
    return [int(m.group(1)) for m in pattern.finditer(text)]

def parse_nr_candidates(path: str) -> list:
    return _parse_counts(path, _SCENARIOS_RE)

def parse_discards(path: str) -> list:
    return _parse_counts(path, _DROPPED_RE)

def _avg(xs):
    return (sum(xs) / len(xs)) if xs else float("nan")

In [ ]:
for data_folder in ["eedi_data", "sciq_data"]:
    reasoning_path = f"manual_inspection/nr_of_candidates/{data_folder}/deepseek_reasoner_naive_reasoner_joint.txt"
    cot_path       = f"manual_inspection/nr_of_candidates/{data_folder}/deepseek_chat_naive_cot_joint.txt"
    discards_path  = f"manual_inspection/discards_valid/{data_folder}/deepseek_reasoner_naive_reasoner_joint.txt"

    reasoning_counts = parse_nr_candidates(reasoning_path)
    cot_counts       = parse_nr_candidates(cot_path)
    drop_counts      = parse_discards(discards_path)

    print(f"== {data_folder} ==")
    print(f"  nr_of_candidates  reasoning: n={len(reasoning_counts):2d}  avg={_avg(reasoning_counts):.2f}  raw={reasoning_counts}")
    print(f"  nr_of_candidates  CoT      : n={len(cot_counts):2d}  avg={_avg(cot_counts):.2f}  raw={cot_counts}")
    print(f"  discards_valid    reasoner : n={len(drop_counts):2d}  avg={_avg(drop_counts):.2f}  raw={drop_counts}")
    print()